# Inspect Consolidated Runoff Datasets

Load and visualize the two runoff source datasets for January 2000.

Sources (see `catalog/variables.yml` → `runoff`):
- ERA5-Land total runoff (`ro`, m/month)
- GLDAS-2.1 NOAH total runoff (`runoff_total = Qs_acc + Qsb_acc`, kg m-2 ≡ mm/month)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import xarray as xr

DATASTORE = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/nhf-datastore")
PROJECT = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets")

TARGET_TIME = "2000-01-15"
TARGET_YEAR = 2000

## Load both datasets

In [ ]:
datasets = {
    "ERA5-Land (ro)": {
        "path": DATASTORE / "era5_land" / f"era5_land_monthly_{TARGET_YEAR}.nc",
        "var": "ro",
        "units": "m/month",
    },
    "GLDAS-2.1 NOAH (runoff_total)": {
        "path": DATASTORE / "gldas_noah_v21_monthly" / "gldas_noah_v21_monthly.nc",
        "var": "runoff_total",
        "units": "kg m-2 (mm/month)",
    },
}

opened = {}
for label, info in datasets.items():
    nc_path = info["path"]
    if not nc_path.exists():
        print(f"SKIP {label}: {nc_path} not found (run fetch first)")
        continue
    ds = xr.open_dataset(nc_path)
    opened[label] = (ds, info)
    print(f"{label}: {list(ds.data_vars)} | time: {ds.time.values[0]} .. {ds.time.values[-1]} | shape: {dict(ds.sizes)}")

## Dataset representations

In [ ]:
for label, (ds, _) in opened.items():
    print(f"{'=' * 60}\n{label}\n{'=' * 60}")
    display(ds)

## Plot January 2000

In [ ]:
n = len(opened)
if n == 0:
    print("No datasets available yet. Run the fetch commands first.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    axes = axes.flatten() if n > 1 else [axes]

    for idx, (label, (ds, info)) in enumerate(opened.items()):
        ax = axes[idx]
        var = info["var"]
        da = ds[var].sel(time=TARGET_TIME, method="nearest")
        actual_time = str(da.time.values)[:10]

        da.plot(ax=ax, cmap="YlGnBu", robust=True)
        ax.set_title(f"{label}\n{actual_time} | {info['units']}", fontsize=11)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

    for idx in range(len(opened), len(axes)):
        axes[idx].set_visible(False)

    fig.suptitle(f"Runoff — nearest to {TARGET_TIME}", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

## Clean up

In [ ]:
for label, (ds, _) in opened.items():
    ds.close()